In [ ]:
ROOT = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical")
OUT  = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical\output_aug"); OUT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)

def add_gaussian_noise(arr, snr_db=18.0, clip_zero=True):
    arr = np.clip(arr, 0, None) if clip_zero else arr
    power = float(np.mean(arr**2)) if arr.size else 0.0
    snr = 10**(snr_db/10)
    sigma = np.sqrt(power / snr) if snr > 0 else 0.0
    out = arr + rng.normal(0.0, sigma, arr.shape)
    return np.clip(out, 0, None) if clip_zero else out

def add_poisson_noise(arr, scale=0.5, clip_zero=True):
    lam = np.maximum(arr * scale, 0.0)
    out = rng.poisson(lam).astype(np.float32) / max(scale, 1e-6)
    return np.clip(out, 0, None) if clip_zero else out

def multiplicative_jitter(arr, sigma=0.05, clip_zero=True):
    out = arr * rng.normal(1.0, sigma, size=arr.shape)
    return np.clip(out, 0, None) if clip_zero else out

def small_axis_shift(arr, rt_px=2, dt_px=2):
    out = arr
    if rt_px != 0:
        out = np.roll(out, rt_px, axis=0)
    if dt_px != 0:
        out = np.roll(out, dt_px, axis=1)
    return out

def process_mea(mea_path: Path, snr_db=18.0, jitter_sigma=0.05, rt_shift=(-2,2), dt_shift=(-2,2)):
    spec = ims.Spectrum.read_mea(str(mea_path))
    M = np.clip(spec.values.astype(np.float32), 0, None)
    rt_px = rng.integers(rt_shift[0], rt_shift[1]+1) if rt_shift[0] <= rt_shift[1] else 0
    dt_px = rng.integers(dt_shift[0], dt_shift[1]+1) if dt_shift[0] <= dt_shift[1] else 0
    A = small_axis_shift(M, rt_px=rt_px, dt_px=dt_px)
    A = multiplicative_jitter(A, sigma=jitter_sigma)
    A = add_gaussian_noise(A, snr_db=snr_db)

    base = mea_path.stem
    np.save(OUT / f"{base}_M.npy", M)
    np.save(OUT / f"{base}_M_aug.npy", A)
    np.save(OUT / f"{base}_rt.npy", spec.ret_time)
    np.save(OUT / f"{base}_dt.npy", spec.drift_time)

    return {
        "file": str(mea_path),
        "shape": f"{M.shape[0]}x{M.shape[1]}",
        "rt_shift_px": int(rt_px),
        "dt_shift_px": int(dt_px),
        "jitter_sigma": float(jitter_sigma),
        "snr_db": float(snr_db),
        "clean_min": float(M.min()),
        "clean_max": float(M.max()),
        "aug_min": float(A.min()),
        "aug_max": float(A.max()),
    }

files = sorted(ROOT.rglob("*.mea"))
log = []
for f in files:
    try:
        log.append(process_mea(f, snr_db=18.0, jitter_sigma=0.05, rt_shift=(-2,2), dt_shift=(-2,2)))
    except Exception as e:
        log.append({"file": str(f), "error": repr(e)})

pd.DataFrame(log).to_csv(OUT / "augmentation_log.csv", index=False)
print(f"Processed {len(files)} files → {OUT}")


In [ ]:
MEA_ROOT  = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical")
DATA_ROOT = Path(r"C:\Users\user\Desktop\Thesis_project\IMS_Honey_Botanical\output_aug")
N = 20

pairs = []
for mea in sorted(MEA_ROOT.rglob("*.mea")):
    stem = mea.stem
    aug = DATA_ROOT / f"{stem}_M_aug.npy"
    if not aug.exists():
        alt = DATA_ROOT / f"{stem}_M_noisy.npy"
        if alt.exists():
            aug = alt
        else:
            continue
    pairs.append((mea, aug))
    if len(pairs) >= N:
        break

print(f"Comparing {len(pairs)} pairs")

def psnr(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    mse = np.mean((x - y) ** 2)
    if mse == 0:
        return float("inf"), 0.0
    peak = np.max(x) - np.min(x)
    return 20 * np.log10(peak / np.sqrt(mse + 1e-12)), mse

for mea_path, aug_path in pairs:
    spec = ims.Spectrum.read_mea(str(mea_path))
    clean = np.clip(spec.values.astype(np.float32), 0, None)
    aug = np.load(aug_path).astype(np.float32)
    vmin, vmax = float(np.quantile(clean, 0.01)), float(np.quantile(clean, 0.99))

    p, mse = psnr(clean, aug)
    print(f"{mea_path.name}  vs  {aug_path.name}  ->  PSNR: {p:.2f} dB, MSE: {mse:.4g}")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    im0 = axes[0].imshow(clean, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap="viridis")
    axes[0].set_title(f"Clean: {mea_path.name}")
    axes[0].set_xlabel("Drift time (index)")
    axes[0].set_ylabel("Retention time (index)")

    im1 = axes[1].imshow(aug, aspect="auto", origin="lower", vmin=vmin, vmax=vmax, cmap="viridis")
    axes[1].set_title(f"Augmented: {aug_path.name}")
    axes[1].set_xlabel("Drift time (index)")

    diff = aug - clean
    dlim = float(np.quantile(np.abs(diff), 0.995))
    im2 = axes[2].imshow(diff, aspect="auto", origin="lower", vmin=-dlim, vmax=dlim, cmap="seismic")
    axes[2].set_title("Augmented − Clean (diff)")
    axes[2].set_xlabel("Drift time (index)")

    cbar = fig.colorbar(im1, ax=axes[:2], shrink=0.8)
    cbar.set_label("Intensity")
    cbar2 = fig.colorbar(im2, ax=[axes[2]], shrink=0.8)
    cbar2.set_label("Δ Intensity")

    plt.show()
